# 🥙 LSTM on Recipe Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DiveshJ8/Generative_Deep_Learning_2nd_Edition/blob/feature/assignment_4/notebooks/05_autoregressive/01_lstm/lstm_modified.ipynb)

In this notebook, we'll walk through the steps required to train your own LSTM on the recipes dataset.

In [30]:
# Setup for Google Colab and local environments
import os
import sys

# Python 3.12+ compatibility fix: 'imp' was removed in Python 3.12+, but legacy extensions may import it
if 'imp' not in sys.modules:
    try:
        import imp
    except ImportError:
        import importlib
        import types
        _imp = types.ModuleType('imp')
        _imp.reload = importlib.reload
        sys.modules['imp'] = _imp

# 1. If running in Google Colab, clone the repository branch and navigate into it
if "google.colab" in sys.modules:
    if os.path.exists("/content"):
        %cd /content
    if not os.path.exists("Generative_Deep_Learning_2nd_Edition"):
        !git clone -b feature/assignment_4 https://github.com/DiveshJ8/Generative_Deep_Learning_2nd_Edition.git
    %cd /content/Generative_Deep_Learning_2nd_Edition
    if os.path.abspath(".") not in sys.path:
        sys.path.append(os.path.abspath("."))
    # Install required packages in Colab
    %pip install -q kaggle pandas kagglehub tensorboard

# 2. Add repository root to sys.path if running locally outside repository root
repo_root = None
for candidate in [
    os.path.abspath(os.path.join(os.getcwd(), "../../..")),
    os.path.abspath(os.getcwd()),
    os.path.abspath("/content/Generative_Deep_Learning_2nd_Edition"),
    "/app",
]:
    if os.path.exists(os.path.join(candidate, "notebooks")):
        repo_root = candidate
        break
if repo_root is None:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

if repo_root not in sys.path:
    sys.path.append(repo_root)

# 3. Ensure required packages are installed
try:
    import pandas
except ImportError:
    %pip install -q pandas

try:
    import tensorflow
except ImportError:
    %pip install -q tensorflow

try:
    import tensorboard
except ImportError:
    %pip install -q tensorboard

/content
/content/Generative_Deep_Learning_2nd_Edition


In [31]:
# Python 3.12+ compatibility shim for 'imp' (used by legacy autoreload)
import sys
if 'imp' not in sys.modules:
    try:
        import imp
    except ImportError:
        import importlib
        import types
        _imp = types.ModuleType('imp')
        _imp.reload = importlib.reload
        sys.modules['imp'] = _imp

# Enable autoreload if available and supported
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass

import os
import json
import re
import string
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, losses

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
tpus = tf.config.list_physical_devices('TPU')
print(f"GPU available: {bool(gpus)}")
if gpus:
    print(f"🚀 Using GPU: {gpus[0].name}")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except (RuntimeError, ValueError):
            pass
elif tpus:
    print(f"⚡ TPU detected: {tpus[0].name}")
    print("ℹ️ Note: For fastest training in Colab, switch to a T4 GPU runtime (Runtime -> Change runtime type -> T4 GPU).")
else:
    tpu_detected = False
    try:
        tf.distribute.cluster_resolver.TPUClusterResolver()
        tpu_detected = True
    except Exception:
        pass
    if tpu_detected:
        print("⚡ Connected to Colab TPU environment.")
        print("ℹ️ Note: For fastest training in Colab, switch to a T4 GPU runtime (Runtime -> Change runtime type -> T4 GPU).")
    else:
        print("⚠️ No GPU detected (running on CPU). In Google Colab, go to: Runtime -> Change runtime type -> T4 GPU (Hardware Accelerator).")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
TensorFlow version: 2.20.0
GPU available: True
🚀 Using GPU: /physical_device:GPU:0


## 0. Parameters <a name="parameters"></a>

In [32]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 25

## 1. Load the data <a name="load"></a>

In [33]:
# Load the full dataset across Docker, local, Kaggle, Drive, or Colab environments
import os
import sys
import shutil
import zipfile
import getpass
import builtins
import json

# Ensure builtins.exit is defined (fixes "name 'exit' is not defined" if Kaggle authentication fails)
if not hasattr(builtins, "exit"):
    builtins.exit = sys.exit
if not hasattr(builtins, "quit"):
    builtins.quit = sys.exit

# 1. Resolve dataset directory across environments
data_dir = None
candidate_dirs = [
    os.path.join(repo_root, "data", "epirecipes"),
    "/app/data/epirecipes",
    os.path.abspath("./data/epirecipes"),
    os.path.abspath("../../../data/epirecipes"),
    "/kaggle/input/epirecipes",
    "/content/data/epirecipes",
    "/content/Generative_Deep_Learning_2nd_Edition/data/epirecipes",
    "/content/drive/MyDrive/epirecipes",
    "/content/drive/MyDrive/data/epirecipes",
]

for candidate in candidate_dirs:
    if os.path.exists(os.path.join(candidate, "full_format_recipes.json")):
        data_dir = candidate
        break

# Check kagglehub cache directory if previously downloaded
if data_dir is None:
    kh_cache_base = os.path.expanduser("~/.cache/kagglehub/datasets/hugodarwood/epirecipes")
    if os.path.exists(kh_cache_base):
        for root, dirs, files in os.walk(kh_cache_base):
            if "full_format_recipes.json" in files:
                data_dir = root
                break

if data_dir is None:
    data_dir = os.path.join(repo_root, "data", "epirecipes")

data_path = os.path.join(data_dir, "full_format_recipes.json")

# 2. If dataset is missing, automatically download and extract
if not os.path.exists(data_path):
    print(f"Dataset not found at '{data_path}'.")
    download_success = False

    # Strategy A: Use kagglehub (allows anonymous download of public datasets without API keys)
    try:
        print("Strategy A: Attempting download via kagglehub (no API key required)...\n")
        import kagglehub
        kh_path = kagglehub.dataset_download("hugodarwood/epirecipes")
        if kh_path and os.path.exists(kh_path):
            for root, dirs, files in os.walk(kh_path):
                if "full_format_recipes.json" in files:
                    data_dir = root
                    data_path = os.path.join(data_dir, "full_format_recipes.json")
                    download_success = True
                    print(f"✅ Successfully downloaded dataset via kagglehub to: {data_dir}")
                    break
                for f in files:
                    if f.endswith(".zip"):
                        with zipfile.ZipFile(os.path.join(root, f), "r") as z:
                            z.extractall(root)
                        if os.path.exists(os.path.join(root, "full_format_recipes.json")):
                            data_dir = root
                            data_path = os.path.join(data_dir, "full_format_recipes.json")
                            download_success = True
                            print(f"✅ Successfully downloaded & extracted dataset via kagglehub to: {data_dir}")
                            break
    except Exception as e_kh:
        print(f"kagglehub download attempt skipped/failed ({e_kh}). Proceeding to Kaggle API...")

    # Strategy B: Use Kaggle API / credentials
    if not download_success:
        print("Strategy B: Attempting download via Kaggle credentials...")

        # Check Google Colab Secrets (Left sidebar -> 🔑 Secrets -> KAGGLE_USERNAME, KAGGLE_KEY)
        if "google.colab" in sys.modules:
            try:
                from google.colab import userdata
                if "KAGGLE_USERNAME" not in os.environ and userdata.get("KAGGLE_USERNAME"):
                    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
                if "KAGGLE_KEY" not in os.environ and userdata.get("KAGGLE_KEY"):
                    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            except Exception:
                pass

        # Check Google Drive for kaggle.json
        drive_kaggle_paths = [
            "/content/drive/MyDrive/kaggle.json",
            "/content/drive/MyDrive/.kaggle/kaggle.json",
            "/content/drive/MyDrive/Kaggle/kaggle.json",
        ]
        for p in drive_kaggle_paths:
            if os.path.exists(p):
                os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
                shutil.copy(p, os.path.expanduser("~/.kaggle/kaggle.json"))
                os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
                break

        # Check repository root for kaggle.json
        if os.path.exists(os.path.join(repo_root, "kaggle.json")):
            os.environ["KAGGLE_CONFIG_DIR"] = repo_root

        # Check if Kaggle credentials exist
        has_creds = (
            os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json"))
            or (os.environ.get("KAGGLE_CONFIG_DIR") and os.path.exists(os.path.join(os.environ["KAGGLE_CONFIG_DIR"], "kaggle.json")))
            or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
        )

        # Interactive fallback prompt if running in Colab without credentials
        if not has_creds and "google.colab" in sys.modules:
            print("\nKaggle credentials not detected.")
            print("To download the epirecipes dataset from Kaggle, choose one:")
            print("  1. Enter your Kaggle Username & API Key")
            print("  2. Upload kaggle.json file")
            print("  3. Skip / cancel")
            try:
                choice = input("Enter choice (1/2/3, default=1): ").strip() or "1"
                if choice == "1":
                    uname = input("Kaggle Username: ").strip()
                    ukey = getpass.getpass("Kaggle API Key: ").strip()
                    if uname and ukey:
                        os.environ["KAGGLE_USERNAME"] = uname
                        os.environ["KAGGLE_KEY"] = ukey
                        has_creds = True
                elif choice == "2":
                    from google.colab import files
                    print("Please upload kaggle.json:")
                    uploaded = files.upload()
                    for fname in uploaded:
                        if fname.endswith(".json"):
                            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
                            shutil.move(fname, os.path.expanduser("~/.kaggle/kaggle.json"))
                            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
                            has_creds = True
                            break
            except Exception as e_input:
                print(f"Interactive prompt skipped: {e_input}")

        if has_creds:
            try:
                from kaggle.api.kaggle_api_extended import KaggleApi
                api = KaggleApi()
                api.authenticate()

                os.makedirs(data_dir, exist_ok=True)
                parent_dir = os.path.dirname(data_dir)
                zip_path = os.path.join(parent_dir, "epirecipes.zip")
                print("Downloading hugodarwood/epirecipes (~35 MB)...\n")
                api.dataset_download_files("hugodarwood/epirecipes", path=parent_dir, unzip=False)
                print("Extracting files...")
                with zipfile.ZipFile(zip_path, "r") as zip_ref:
                    zip_ref.extractall(data_dir)
                if os.path.exists(zip_path):
                    os.remove(zip_path)
                download_success = True
                print("✅ Download and extraction complete!")
            except Exception as e_api:
                print(f"Kaggle API download failed: {e_api}")

    # If still not found, raise clear actionable error
    if not os.path.exists(data_path):
        raise FileNotFoundError(
            f"Could not find 'full_format_recipes.json' at '{data_path}'.\n\n"
            f"To provide the Recipes dataset, please use one of these quick options:\n"
            f"1. In Google Colab, add your Kaggle credentials to Secrets (key icon 🔑 on left sidebar):\n"
            f"   - Name: KAGGLE_USERNAME, Value: <your_kaggle_username>\n"
            f"   - Name: KAGGLE_KEY, Value: <your_kaggle_api_key>\n"
            f"   (Generate key at: https://www.kaggle.com/settings -> 'Create New Token')\n\n"
            f"2. Or upload 'kaggle.json' to Colab / repository root.\n\n"
            f"3. Or run the shell downloader script:\n"
            f"   bash scripts/downloaders/download_kaggle_data.sh hugodarwood epirecipes\n"
        )

# Load the full dataset
with open(data_path, "r", encoding="utf-8") as json_data:
    recipe_data = json.load(json_data)

print(f"Successfully loaded {len(recipe_data)} raw recipes from: {data_path}")

Successfully loaded 20130 raw recipes from: /kaggle/input/epirecipes/full_format_recipes.json


In [34]:
# Filter the dataset
filtered_data = [
    "Recipe for " + x["title"] + " | " + " ".join(x["directions"])
    for x in recipe_data
    if "title" in x
    and x["title"] is not None
    and "directions" in x
    and x["directions"] is not None
]

In [35]:
# Count the recipes
n_recipes = len(filtered_data)
print(f"{n_recipes} recipes loaded")

20111 recipes loaded


In [36]:
example = filtered_data[9]
print(example)

Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas  | Chop enough parsley leaves to measure 1 tablespoon; reserve. Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan, covered, 5 minutes. Meanwhile, sprinkle gelatin over water in a medium bowl and let soften 1 minute. Strain broth through a fine-mesh sieve into bowl with gelatin and stir to dissolve. Season with salt and pepper. Set bowl in an ice bath and cool to room temperature, stirring. Toss ham with reserved parsley and divide among jars. Pour gelatin on top and chill until set, at least 1 hour. Whisk together mayonnaise, mustard, vinegar, 1/4 teaspoon salt, and 1/4 teaspoon pepper in a large bowl. Stir in celery, cornichons, and potatoes. Pulse peas with marjoram, oil, 1/2 teaspoon pepper, and 1/4 teaspoon salt in a food processor to a coarse mash. Layer peas, then potato salad, over ham.


## 2. Tokenise the data

In [37]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [38]:
# Display an example of a recipe
example_data = text_data[9]
example_data

'Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas | Chop enough parsley leaves to measure 1 tablespoon ; reserve . Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan , covered , 5 minutes . Meanwhile , sprinkle gelatin over water in a medium bowl and let soften 1 minute . Strain broth through a fine - mesh sieve into bowl with gelatin and stir to dissolve . Season with salt and pepper . Set bowl in an ice bath and cool to room temperature , stirring . Toss ham with reserved parsley and divide among jars . Pour gelatin on top and chill until set , at least 1 hour . Whisk together mayonnaise , mustard , vinegar , 1 / 4 teaspoon salt , and 1 / 4 teaspoon pepper in a large bowl . Stir in celery , cornichons , and potatoes . Pulse peas with marjoram , oil , 1 / 2 teaspoon pepper , and 1 / 4 teaspoon salt in a food processor to a coarse mash . Layer peas , then potato salad , over ham . '

In [39]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [40]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [41]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [42]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: .
3: ,
4: and
5: to
6: in
7: the
8: with
9: a


In [43]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[  26   16  557    1    8  298  335  189    4 1054  494   27  332  228
  235  262    5  594   11  133   22  311    2  332   45  262    4  671
    4   70    8  171    4   81    6    9   65   80    3  121    3   59
   12    2  299    3   88  650   20   39    6    9   29   21    4   67
  529   11  164    2  320  171  102    9  374   13  643  306   25   21
    8  650    4   42    5  931    2   63    8   24    4   33    2  114
   21    6  178  181 1245    4   60    5  140  112    3   48    2  117
  557    8  285  235    4  200  292  980    2  107  650   28   72    4
  108   10  114    3   57  204   11  172    2   73  110  482    3  298
    3  190    3   11   23   32  142   24    3    4   11   23   32  142
   33    6    9   30   21    2   42    6  353    3 3224    3    4  150
    2  437  494    8 1281    3   37    3   11   23   15  142   33    3
    4   11   23   32  142   24    6    9  291  188    5    9  412  572
    2  230  494    3   46  335  189    3   20  557    2    0    0    0
    0 

## 3. Create the Training Set

In [44]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_ds = (
    text_ds.map(prepare_inputs, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

## 4. Build the LSTM <a name="build"></a>

In [45]:
inputs = layers.Input(shape=(None,), dtype="int32")
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
lstm = models.Model(inputs, outputs)
lstm.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 100)      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, None, 128)      │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 10000)    │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,407,248 (9.18 MB)

 Trainable params: 2,407,248 (9.18 MB)

 Non-trainable params: 0 (0.00 B)

In [46]:
if LOAD_MODEL:
    import os
    candidate_models = [
        "./models_modified/lstm.keras",
        "./models_modified/lstm",
        "./models/lstm.keras",
        "./models/lstm",
    ]
    model_path = next((p for p in candidate_models if os.path.exists(p)), None)
    if model_path:
        print(f"Loading model from {model_path}...")
        lstm = models.load_model(model_path, compile=False)
    else:
        candidate_weights = [
            "./checkpoint_modified/checkpoint.weights.h5",
            "./checkpoint/checkpoint.weights.h5",
            "./checkpoint/checkpoint.ckpt",
        ]
        weight_path = next((p for p in candidate_weights if os.path.exists(p)), None)
        if weight_path:
            print(f"Loading weights from {weight_path}...")
            lstm.load_weights(weight_path)
        else:
            print("No existing model or weights found. Starting with initial weights.")

## 5. Train the LSTM <a name="train"></a>

In [47]:
loss_fn = losses.SparseCategoricalCrossentropy()
lstm.compile("adam", loss_fn)

In [48]:
# Enhanced TextGenerator callback with Top-k, Nucleus (Top-p) sampling, repetition penalty, and detokenization
def detokenize(text):
    """Clean raw tokenized text into proper culinary sentences."""
    text = re.sub(r'(\d+)\s*/\s*(\d+)', r'\1/\2', text)
    text = re.sub(r'\s+([,.:;!?"\'])', r'\1', text)
    text = re.sub(r'\(\s+', r'(', text)
    text = re.sub(r'\s+\)', r')', text)
    text = re.sub(r'\s+-\s+', r'-', text)
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = text.split('. ')
    sentences = [s[0].upper() + s[1:] if len(s) > 1 else s.upper() for s in sentences]
    return '. '.join(sentences)


class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=40, top_p=0.9, repetition_penalty=1.1, model=None):
        super(TextGenerator, self).__init__()
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }
        self.top_k = top_k
        self.top_p = top_p
        self.repetition_penalty = repetition_penalty
        if model is not None:
            self.set_model(model)

    def sample_from(self, probs, temperature, recent_tokens=None):
        temperature = max(float(temperature), 1e-5)
        probs = np.asarray(probs, dtype=np.float64)
        probs = np.clip(probs, 1e-12, 1.0)
        
        # Apply repetition penalty to recent tokens to prevent autoregressive looping
        if self.repetition_penalty > 1.0 and recent_tokens:
            for tok in set(recent_tokens[-20:]):
                if tok < len(probs):
                    probs[tok] /= self.repetition_penalty

        # Temperature scaling
        logits = np.log(probs) / temperature
        exp_logits = np.exp(logits - np.max(logits))
        probs = exp_logits / np.sum(exp_logits)

        # Top-k filtering: remove noise from the tail of 10,000 vocabulary words
        if self.top_k is not None and self.top_k > 0:
            top_k = min(self.top_k, len(probs))
            top_indices = np.argsort(probs)[-top_k:]
            filtered_probs = np.zeros_like(probs)
            filtered_probs[top_indices] = probs[top_indices]
            probs = filtered_probs / np.sum(filtered_probs)

        # Top-p (nucleus) filtering: dynamically constrain to cumulative mass p
        if self.top_p is not None and 0.0 < self.top_p < 1.0:
            sorted_indices = np.argsort(probs)[::-1]
            sorted_probs = probs[sorted_indices]
            cumulative_probs = np.cumsum(sorted_probs)
            cutoff = cumulative_probs > self.top_p
            cutoff[0] = False
            sorted_probs[cutoff] = 0.0
            sorted_probs /= np.sum(sorted_probs)
            probs = np.zeros_like(probs)
            probs[sorted_indices] = sorted_probs

        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        gen_model = getattr(self, "model", None)
        if gen_model is None:
            raise ValueError("Model is not assigned to TextGenerator. Please pass model or run lstm.fit().")

        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y = gen_model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature, recent_tokens=start_tokens)
            info.append({"prompt": start_prompt, "word_probs": probs})
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]

        print(f"\ngenerated text:\n{start_prompt}\n")
        print(f"formatted text:\n{detokenize(start_prompt)}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("recipe for", max_tokens=100, temperature=0.7)

In [49]:
# Create directories for checkpoints, logs, and output
import os
os.makedirs("./checkpoint_modified", exist_ok=True)
os.makedirs("./logs_modified", exist_ok=True)
os.makedirs("./models_modified", exist_ok=True)
os.makedirs("./output_modified", exist_ok=True)

# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint_modified/checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = None
try:
    import tensorboard  # noqa: F401
    tensorboard_callback = callbacks.TensorBoard(log_dir="./logs_modified")
except Exception as e:
    print(f"Notice: TensorBoard logging skipped ({e}).")

# Initialize TextGenerator with vocab and bind to model
text_generator = TextGenerator(vocab, model=lstm)

In [50]:
callbacks_list = [model_checkpoint_callback, text_generator]
if tensorboard_callback is not None:
    callbacks_list.insert(1, tensorboard_callback)

lstm.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list,
)

Epoch 1/25
629/629 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 4.9834
generated text:
recipe for grilled with lemon oven | preheat oven with a saucepan . add a medium bowl and add mixture to salt , heat and bring until be reduced is golden is just tender . ) to medium bowl , and 1 / 2 - high heat to a medium bowl over medium - small bowl . add a high , 3 - small saucepan , heat until a very cooked to tender , about 15 minutes . season to a 1 / 2 cup to 4 - medium - simmer , and 1 / 4 with large bowl .

formatted text:
Recipe for grilled with lemon oven | preheat oven with a saucepan. Add a medium bowl and add mixture to salt, heat and bring until be reduced is golden is just tender.) to medium bowl, and 1/2-high heat to a medium bowl over medium-small bowl. Add a high, 3-small saucepan, heat until a very cooked to tender, about 15 minutes. Season to a 1/2 cup to 4-medium-simmer, and 1/4 with large bowl.

629/629 ━━━━━━━━━━━━━━━━━━━━ 38s 58ms/step - loss: 4.1312
Epoch 2/25
628/629 ━━━━━━━

In [51]:
# Save the final model
import os
os.makedirs("./models_modified", exist_ok=True)
lstm.save("./models_modified/lstm.keras")

# Also save to ./models for backward compatibility
os.makedirs("./models", exist_ok=True)
try:
    lstm.save("./models/lstm.keras")
except Exception:
    pass

print("✅ Model successfully saved to ./models_modified/lstm.keras")

✅ Model successfully saved to ./models_modified/lstm.keras


## 6. Generate text using the LSTM

In this section, we evaluate the generative network across **two distinct temperature values ($T = 0.5$ and $T = 0.8$)** to assess quality, coherence, and lexical creativity without the extreme underflow of $0.2$ or chaotic divergence of $1.0$.

In [52]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, idx in zip(p_sorted, i_sorted):
            print(f"{vocab[idx]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [53]:
# Generate text for Prompt 1 at Temperature = 0.5 (Balanced: high syntactic coherence)
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=0.5
)


generated text:
recipe for roasted vegetables | chop 1 / 2 cup

formatted text:
Recipe for roasted vegetables | chop 1/2 cup



In [54]:
print_probs(info, vocab)


PROMPT: recipe for roasted vegetables | chop 1 /
2:   	100.0%
pleating:   	0.0%
pry:   	0.0%
pulses:   	0.0%
ranch:   	0.0%
--------


PROMPT: recipe for roasted vegetables | chop 1 / 2
cup:   	100.0%
petals:   	0.0%
piles:   	0.0%
pleating:   	0.0%
pry:   	0.0%
--------



In [55]:
# Generate text for Prompt 1 at Temperature = 0.8 (Creative: rich culinary variety)
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=0.8
)


generated text:
recipe for roasted vegetables | chop 1 / 2 cup

formatted text:
Recipe for roasted vegetables | chop 1/2 cup



In [56]:
print_probs(info, vocab)


PROMPT: recipe for roasted vegetables | chop 1 /
2:   	100.0%
pleating:   	0.0%
pry:   	0.0%
pulses:   	0.0%
ranch:   	0.0%
--------


PROMPT: recipe for roasted vegetables | chop 1 / 2
cup:   	92.06%
teaspoon:   	7.94%
piles:   	0.0%
pleating:   	0.0%
pry:   	0.0%
--------



In [57]:
# Generate text for Prompt 2 at Temperature = 0.5
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=7, temperature=0.5
)
print_probs(info, vocab)


generated text:
recipe for chocolate ice cream | in

formatted text:
Recipe for chocolate ice cream | in


PROMPT: recipe for chocolate ice cream |
in:   	70.14%
bring:   	24.73%
cook:   	5.13%
piles:   	0.0%
pleating:   	0.0%
--------



In [58]:
# Generate text for Prompt 2 at Temperature = 0.8
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=7, temperature=0.8
)
print_probs(info, vocab)


generated text:
recipe for chocolate ice cream | stir

formatted text:
Recipe for chocolate ice cream | stir


PROMPT: recipe for chocolate ice cream |
in:   	43.81%
bring:   	22.83%
cook:   	8.54%
combine:   	6.52%
stir:   	4.23%
--------



In [59]:
# Extended Recipe Continuation: Temperature 0.5 vs 0.8 on Prompt 1
print("=== ROASTED VEGETABLES (Temperature = 0.5) ===")
_ = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=70, temperature=0.5
)

print("\n=== ROASTED VEGETABLES (Temperature = 0.8) ===")
_ = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=70, temperature=0.8
)

=== ROASTED VEGETABLES (Temperature = 0.5) ===

generated text:
recipe for roasted vegetables | chop 1 / 2 cup cold water . place tomatillos in blender . place colander in large bowl . add 1 / 2 cup water to cover ; toss to blend . ( can be prepared 1 day ahead . cover and refrigerate . ) preheat oven to 400°f . bake potatoes until tender and golden , about 10 minutes . transfer to rack and

formatted text:
Recipe for roasted vegetables | chop 1/2 cup cold water. Place tomatillos in blender. Place colander in large bowl. Add 1/2 cup water to cover; toss to blend. (can be prepared 1 day ahead. Cover and refrigerate.) preheat oven to 400°f. Bake potatoes until tender and golden, about 10 minutes. Transfer to rack and


=== ROASTED VEGETABLES (Temperature = 0.8) ===

generated text:
recipe for roasted vegetables | chop 1 / 2 cup boiling water and drain . melt butter in a 12 - inch heavy skillet over moderately high heat until hot but not smoking , then brown on all sides , about 10 minute

In [60]:
# Extended Recipe Continuation: Temperature 0.5 vs 0.8 on Prompt 2
print("=== CHOCOLATE ICE CREAM (Temperature = 0.5) ===")
_ = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=70, temperature=0.5
)

print("\n=== CHOCOLATE ICE CREAM (Temperature = 0.8) ===")
_ = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=70, temperature=0.8
)

=== CHOCOLATE ICE CREAM (Temperature = 0.5) ===

generated text:
recipe for chocolate ice cream | bring cream and sugar to boil in heavy medium saucepan over medium heat , stirring until sugar dissolves . remove from heat . whisk in cream and vanilla . add 1 cup ice cream maker ; process until smooth . strain into large bowl . add enough water to measure 2 cups water to cover . bring to boil . reduce heat to medium

formatted text:
Recipe for chocolate ice cream | bring cream and sugar to boil in heavy medium saucepan over medium heat, stirring until sugar dissolves. Remove from heat. Whisk in cream and vanilla. Add 1 cup ice cream maker; process until smooth. Strain into large bowl. Add enough water to measure 2 cups water to cover. Bring to boil. Reduce heat to medium


=== CHOCOLATE ICE CREAM (Temperature = 0.8) ===

generated text:
recipe for chocolate ice cream | cook strawberries and sugar in a small saucepan over medium heat , stirring until sugar is dissolved . add cream and cr